In [ ]:
# Désinstaller la version problématique et installer la bonne version
!pip uninstall -y crewai crewai-tools
!pip install crewai==0.86.0 crewai-tools==0.12.1
!pip install langchain langchain-groq langchain-community langchain-core langgraph faiss-cpu langchain-huggingface sentence-transformers

Found existing installation: crewai 1.10.0
Uninstalling crewai-1.10.0:
  Successfully uninstalled crewai-1.10.0
Found existing installation: crewai-tools 1.10.0
Uninstalling crewai-tools-1.10.0:
  Successfully uninstalled crewai-tools-1.10.0
INFO: pip is looking at multiple versions of crewai to determine which version is compatible with other requirements. This could take a while.
ERROR: Cannot install crewai-tools==0.12.1 and crewai==0.86.0 because these package versions have conflicting dependencies.

The conflict is caused by:
    The user requested crewai-tools==0.12.1
    crewai 0.86.0 depends on crewai-tools>=0.17.0

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove package versions to allow pip to attempt to solve the dependency conflict

ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [ ]:
!pip install -U langchain-huggingface

In [5]:
from typing import TypedDict, List, Annotated
import operator
import json
import os
import re
from crewai import LLM

# Imports LangChain & LangGraph
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.graph import StateGraph, END

# Imports CrewAI
from crewai import Agent, Task, Crew, Process

# ===============================
# CONFIGURATION & SÉCURITÉ
# ===============================
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# ✅ MODÈLES GROQ VALIDES (mis à jour 2026)
VALID_GROQ_MODELS = {
    "small": "llama-3.1-8b-instant",          # Rapide et efficace
    "large": "llama-3.3-70b-versatile",       # llama-3.1-70b
    "mix": "mixtral-8x7b-32768",               # Alternative puissante
    "gemma": "gemma2-9b-it"                    # Option supplémentaire
}

# ===============================
# 1. STATE (LangGraph)
# ===============================
class AgentState(TypedDict):
    user_query: str
    routing_decision: List[str]
    text_evidence: Annotated[List[str], operator.add]
    graph_evidence: str
    image_evidence: str
    final_answer: str
    iteration_count: int

# ===============================
# 2. LLM SETUP AVEC GESTION D'ERREURS
# ===============================
def create_llm(model_name: str, temperature: float = 0.7):
    """Crée un LLM avec modèle valide et fallback automatique"""
    try:
        llm = ChatGroq(
            groq_api_key=GROQ_API_KEY,
            model_name=model_name,
            temperature=temperature,
            max_retries=2,
            timeout=30
        )
        # Test de validation
        llm.invoke("test")
        return llm
    except Exception as e:
        print(f"⚠️ Erreur avec {model_name}: {e}")
        print(f"🔄 Fallback vers {VALID_GROQ_MODELS['small']}")
        return ChatGroq(
            groq_api_key=GROQ_API_KEY,
            model_name=VALID_GROQ_MODELS["small"],
            temperature=temperature
        )

# ===============================
# 3. VECTOR STORE
# ===============================
class DocumentStore:
    def __init__(self, documents: List[str]):
        print("📚 Chargement des embeddings...")
        self.embeddings = HuggingFaceEmbeddings(
            model_name="all-MiniLM-L6-v2",
            model_kwargs={'device': 'cpu'}
        )
        docs = [Document(page_content=doc) for doc in documents]
        self.vectorstore = FAISS.from_documents(docs, self.embeddings)
        print(f"✅ {len(documents)} documents indexés")

    def similarity_search(self, query: str, k: int = 3) -> List[str]:
        results = self.vectorstore.similarity_search(query, k=k)
        return [doc.page_content for doc in results]

# ===============================
# 4. ROUTER NODE AMÉLIORÉ
# ===============================
def router_node(state: AgentState) -> AgentState:
    """Router avec parsing JSON robuste et fallback"""
    query = state["user_query"].lower()

    llm = create_llm(VALID_GROQ_MODELS["small"], temperature=0)

    prompt = ChatPromptTemplate.from_template("""
Tu es un routeur intelligent. Analyse cette requête et retourne UNIQUEMENT un JSON valide.

Format attendu: ["retriever"] ou ["retriever", "graph"] ou ["retriever", "image"]

Règles:
- TOUJOURS inclure "retriever" pour recherche documentaire
- Ajoute "graph" si: graphique, courbe, tendance, évolution, statistique
- Ajoute "image" si: image, photo, schéma, figure, diagramme

Requête: {query}

Réponds UNIQUEMENT avec le JSON (exemple: ["retriever", "graph"]):
""")

    try:
        chain = prompt | llm
        response = chain.invoke({"query": query})
        content = response.content.strip()

        # Nettoyage du contenu
        content = re.sub(r'```json\s*', '', content)
        content = re.sub(r'```\s*', '', content)
        content = content.strip()

        # Parsing JSON
        routing = json.loads(content)

        # Validation
        if not isinstance(routing, list) or not routing:
            raise ValueError("Le routing doit être une liste non-vide")

    except Exception as e:
        print(f"⚠️ Erreur parsing JSON: {e}")
        # Fallback intelligent basé sur des mots-clés
        routing = ["retriever"]
        if any(word in query for word in ["graph", "graphique", "courbe", "tendance", "évolution", "statistique"]):
            routing.append("graph")
        if any(word in query for word in ["image", "photo", "schéma", "figure", "diagramme"]):
            routing.append("image")

    state["routing_decision"] = routing
    print(f"🔀 ROUTER: {routing}")
    return state

# ===============================
# 5. RETRIEVER NODE
# ===============================
def retriever_node(state: AgentState, vector_store: DocumentStore) -> AgentState:
    """Recherche dans les documents"""
    if "retriever" not in state["routing_decision"]:
        print("⏭️ RETRIEVER: Skippé (non requis)")
        return state

    try:
        results = vector_store.similarity_search(state["user_query"], k=3)
        state["text_evidence"] = results
        print(f"📄 RETRIEVER: {len(results)} documents trouvés")
        for i, doc in enumerate(results, 1):
            print(f"   {i}. {doc[:80]}...")
    except Exception as e:
        print(f"❌ Erreur RETRIEVER: {e}")
        state["text_evidence"] = []

    return state

# ===============================
# 6. CREWAI SYNTHESIS CORRIGÉ
# ===============================
def synthesize_with_crew(state: AgentState, vector_store: DocumentStore) -> AgentState:
    """Synthèse avec CrewAI utilisant le modèle VALIDE"""

    # ✅ CORRECTION: Utilisation du modèle valide
    llm = LLM(
        model=VALID_GROQ_MODELS["large"],  # llama-3.3-70b-versatile
        api_key=GROQ_API_KEY,
        base_url="https://api.groq.com/openai/v1"
    )

    evidence = "\n".join(state["text_evidence"]) if state["text_evidence"] else "Aucune donnée trouvée."

    print("\n🤖 Création de l'équipe CrewAI...")

    # Agents
    data_analyst = Agent(
        role="Analyste de Données Senior",
        goal="Extraire et analyser les informations pertinentes avec précision",
        backstory="Expert en analyse de données avec 10 ans d'expérience.",
        llm=llm,
        verbose=False,
        allow_delegation=False
    )

    synthesizer = Agent(
        role="Synthétiseur Expert",
        goal="Créer des réponses claires, structurées et actionnables",
        backstory="Spécialiste en communication analytique et reporting exécutif.",
        llm=llm,
        verbose=False,
        allow_delegation=False
    )

    # Tasks
    task1 = Task(
        description=f"""
Analyse ces preuves pour répondre à la question suivante:
{state['user_query']}

Preuves disponibles:
{evidence}

Identifie:
- Les chiffres clés
- Les tendances principales
- Les insights importants
        """,
        expected_output="Liste structurée des insights clés avec chiffres précis",
        agent=data_analyst
    )

    task2 = Task(
        description="""
À partir de l'analyse précédente, produis une réponse finale structurée avec:

1. **Synthèse**: Résumé des points clés en 2-3 phrases
2. **Analyse détaillée**: Développement avec chiffres et contexte
3. **Conclusion**: Recommandations ou observations finales

Format: Markdown avec sections claires.
        """,
        expected_output="Réponse structurée en markdown avec sections numérotées",
        agent=synthesizer,
        context=[task1]
    )

    # Crew
    crew = Crew(
        agents=[data_analyst, synthesizer],
        tasks=[task1, task2],
        process=Process.sequential,
        verbose=False
    )

    try:
        print("⚙️ Exécution de CrewAI...")
        result = crew.kickoff()
        state["final_answer"] = str(result)
        print("✅ Synthèse CrewAI terminée")
    except Exception as e:
        print(f"❌ Erreur CrewAI: {e}")
        # Fallback: réponse manuelle
        state["final_answer"] = f"""
## Réponse basée sur les preuves disponibles

**Question**: {state['user_query']}

**Données analysées**:
{evidence}

**Synthèse**: L'analyse des documents révèle les informations ci-dessus.
Une erreur technique a empêché la synthèse avancée, mais les données brutes sont présentées.
        """

    return state

# ===============================
# 7. WORKFLOW CONSTRUCTION
# ===============================
def create_workflow(vector_store: DocumentStore):
    """Construit le graphe LangGraph"""
    workflow = StateGraph(AgentState)

    # Ajout des nœuds
    workflow.add_node("router", router_node)
    workflow.add_node("retriever", lambda state: retriever_node(state, vector_store))
    workflow.add_node("synthesizer", lambda state: synthesize_with_crew(state, vector_store))

    # Définition du flux
    workflow.set_entry_point("router")
    workflow.add_edge("router", "retriever")
    workflow.add_edge("retriever", "synthesizer")
    workflow.add_edge("synthesizer", END)

    return workflow.compile()

# ===============================
# 8. MAIN EXECUTION
# ===============================
def main():
    print("=" * 60)
    print("🚀 SYSTÈME MULTI-AGENTS AVANCÉ")
    print("=" * 60)

    # Documents exemple (enrichis)
    documents = [
        "Les ventes ont augmenté de 20% en 2023 grâce à une stratégie marketing digitale efficace.",
        "Le rapport indique une forte croissance au Q2 avec 15% de hausse du chiffre d'affaires.",
        "Les statistiques montrent une tendance positive sur tous les segments de marché.",
        "Le département R&D a investi 5M EUR dans de nouveaux projets d'innovation technologique.",
        "La satisfaction client a atteint 92%, un record historique pour l'entreprise.",
        "Les coûts opérationnels ont diminué de 8% grâce à l'automatisation des processus.",
        "L'expansion en Asie a généré 30% de revenus supplémentaires au Q3."
    ]

    # 1. Initialisation du Vector Store
    print("\n📚 Initialisation du Vector Store...")
    vector_store = DocumentStore(documents)

    # 2. Création du Workflow
    print("\n🔧 Construction du Workflow LangGraph...")
    app = create_workflow(vector_store)
    print("✅ Workflow compilé avec succès")

    # 3. État initial
    initial_state = {
        "user_query": "Analyse les performances des ventes et donne-moi les chiffres clés.",
        "routing_decision": [],
        "text_evidence": [],
        "graph_evidence": "",
        "image_evidence": "",
        "final_answer": "",
        "iteration_count": 0
    }

    # 4. Exécution du Workflow
    print("\n⚡ Exécution du Workflow...")
    try:
        final_state = app.invoke(initial_state)

        print("\n" + "=" * 60)
        print("💡 RÉPONSE FINALE")
        print("=" * 60)
        print(final_state["final_answer"])

    except Exception as e:
        print(f"\n❌ Erreur lors de l'exécution du workflow: {e}")
        import traceback
        traceback.print_exc()

    print("\n" + "=" * 60)
    print("🏁 FIN DU PROGRAMME")
    print("=" * 60)

if __name__ == "__main__":
    main()

🚀 SYSTÈME MULTI-AGENTS AVANCÉ

📚 Initialisation du Vector Store...
📚 Chargement des embeddings...
✅ 7 documents indexés

🔧 Construction du Workflow LangGraph...
✅ Workflow compilé avec succès

⚡ Exécution du Workflow...
🔀 ROUTER: ['retriever', 'graph']
📄 RETRIEVER: 3 documents trouvés
   1. Les ventes ont augmenté de 20% en 2023 grâce à une stratégie marketing digitale ...
   2. Les coûts opérationnels ont diminué de 8% grâce à l'automatisation des processus...
   3. Le rapport indique une forte croissance au Q2 avec 15% de hausse du chiffre d'af...

🤖 Création de l'équipe CrewAI...
⚙️ Exécution de CrewAI...
✅ Synthèse CrewAI terminée

💡 RÉPONSE FINALE
## 1. **Synthèse**
L'analyse des performances des ventes met en évidence une augmentation significative des ventes de 20% en 2023, ainsi qu'une réduction des coûts opérationnels de 8% grâce à l'automatisation des processus. Ces résultats positifs sont également caractérisés par une hausse du chiffre d'affaires au Q2 de 15%, indiquant une

In [ ]:
!pip install -U crewai langchain langgraph langchain-groq langchain-huggingface faiss-cpu